In [1]:
# ======================= IMPORTS =======================
import re, io, requests, pandas as pd, numpy as np, GEOparse as gp, tempfile
from pathlib import Path
from typing import Optional, Tuple, Dict, List, Iterable
from collections import defaultdict
import mygene
import os, shutil
import hashlib

# ======================= CONFIG =======================
USER_AGENT = "alalvaromeca@gmail.com"  # contacto recomendado por NCBI
_GEO_DL = "https://www.ncbi.nlm.nih.gov/geo/download/"

CACHE_DIR = Path("./cache_geo").resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ======================= HTTP helpers =======================
def _http_get(url: str, params=None, stream=False, timeout=(10, 600)):
    return requests.get(url, params=params, stream=stream, timeout=timeout,
                        headers={"User-Agent": USER_AGENT})

def _safe_name(s: str) -> str:
    # nombre de fichero estable y corto
    h = hashlib.md5(s.encode("utf-8")).hexdigest()[:10]
    base = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)[:80]
    return f"{base}__{h}"

def _cache_path(kind: str, key: str, suffix: str):
    return CACHE_DIR / kind / f"{_safe_name(key)}{suffix}"

def _ensure_dir(p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)
    
def _download_bytes(url: str, params=None, timeout=(10, 600), use_cache=True) -> bytes:
    key = url
    if params:
        # orden estable
        key += "?" + "&".join([f"{k}={params[k]}" for k in sorted(params)])

    cache_file = _cache_path("downloads", key, ".bin")
    _ensure_dir(cache_file)

    if use_cache and cache_file.exists() and cache_file.stat().st_size > 0:
        return cache_file.read_bytes()

    with _http_get(url, params=params, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        data = b"".join(r.iter_content(1024 * 1024))

    if use_cache:
        cache_file.write_bytes(data)

    return data


def _is_url(s: str) -> bool:
    return isinstance(s, str) and s.lower().startswith(("http://", "https://", "ftp://"))

def _guess_sep_by_name(name: str) -> str:
    name = name.lower()
    return "," if (name.endswith(".csv") or name.endswith(".csv.gz")) else "\t"

def _find_ensembl_col(df_raw: pd.DataFrame) -> Optional[str]:
    """Devuelve la 1ª columna que parezca mayoritariamente ENSG."""
    for c in df_raw.columns:
        s = df_raw[c].astype(str).head(2000)
        m = s.str.match(r'^ENSG\d+(?:\.\d+)?$')
        if m.mean() > 0.6:
            return c
    return None

def _extract_ensembl_from_index(idx: pd.Index) -> Tuple[pd.Index, float]:
    """Extrae el primer ENSG presente en cada string del índice. Devuelve (nuevo_index, fracción_con_ENSG)."""
    pat = re.compile(r'(ENSG\d+(?:\.\d+)?)')
    vals, hits = [], 0
    for x in map(str, idx):
        m = pat.search(x)
        if m:
            vals.append(m.group(1)); hits += 1
        else:
            vals.append(None)
    frac = hits / max(1, len(idx))
    return pd.Index(vals, name="GeneID"), frac

def _pick_id_col(df: pd.DataFrame) -> str:
    """Detector robusto de columna ID."""
    cols = list(df.columns)
    exact = ["GeneID","gene_id","Gene_Id","geneid","Gene","gene","Symbol","symbol","ID","id",
             "Gene.Name","gene.name","Gene_Symbol","gene_symbol","GeneSymbol","genename","GeneName"]
    for c in exact:
        if c in cols:
            return c
    def norm(s): return re.sub(r"[ _\.\-]+", "", str(s).strip().lower())
    normed = {c: norm(c) for c in cols}
    for c, n in normed.items():
        if n in {"geneid","gene_id"}:
            return c
    for c, n in normed.items():
        if "gene" in n and any(x in n for x in ["id","name","symbol"]):
            return c
    return cols[0]

def enterztpsymbol(df):
    df = df.copy()
    df.index = df.index.astype(str).str.strip()   # asegúrate de que sean strings
    
    mg = mygene.MyGeneInfo()
    # Consulta: tus IDs → símbolo y nombre (humano). Cambia species si no es Homo sapiens.
    hits = mg.querymany(
        df.index.tolist(),
        scopes="entrezgene",
        fields="symbol,name,ensembl.gene",
        species="human",
        as_dataframe=True
    )
    
    # quedarnos con una fila por ID de entrada
    hits = hits[~hits.index.duplicated(keep="first")]
    
    # unir a la expresión
    df_mapped = df.join(hits[["symbol"]], how="left")
    df_mapped.index.name = "entrez_id"
    df = (
        df_mapped
          .dropna(subset=['symbol'])   
          .set_index('symbol')         
    )  
    return df
# ======================= DISCOVERY GEO (type=rnaseq_counts) =======================
def _find_rnaseq_counts_file(gse_id: str, timeout=60) -> Optional[str]:
    """Localiza un .tsv/.txt (.gz) en la página 'type=rnaseq_counts' y prioriza 'raw_counts'."""
    params = {"acc": gse_id, "format": "file", "type": "rnaseq_counts"}
    r = _http_get(_GEO_DL, params=params, timeout=timeout)
    r.raise_for_status()
    html = r.text
    cand = re.findall(r'href="[^"]*?file=([^"&]+?\.(?:tsv|txt)\.gz)"', html, flags=re.I)
    if not cand:
        return None
    raw = [c for c in cand if re.search(r'raw[_-]?counts', c, flags=re.I)]
    return raw[0] if raw else cand[0]

# ======================= Lectores de counts =======================
def _read_counts_from_bytes(data: bytes, filename_hint: str) -> pd.DataFrame:
    compression = "gzip" if filename_hint.lower().endswith(".gz") else None
    sep = _guess_sep_by_name(filename_hint)

    df_raw = pd.read_csv(io.BytesIO(data), sep=sep, header=0,
                         low_memory=False, compression=compression)

    # 1) columna de IDs (robusto)
    id_col = _pick_id_col(df_raw)

    # 2) si NO parece ENSG, intenta encontrar otra columna ENSG
    tmp_idx = df_raw[id_col].astype(str)
    looks_ensg = tmp_idx.str.match(r'^ENSG\d+(?:\.\d+)?$').mean() > 0.6
    if not looks_ensg:
        ensg_col = _find_ensembl_col(df_raw)
        if ensg_col is not None:
            id_col = ensg_col

    # 3) fija índice y convierte numérico (mantén sólo columnas con algo numérico)
    df_raw = df_raw.set_index(id_col)
    num = df_raw.apply(pd.to_numeric, errors="coerce")
    keep = num.columns[~num.isna().all()]
    df = num[keep]

    # 4) limpieza
    df = df.dropna(axis=0, how="all")  # elimina filas 100% NaN (cabeceras tipo 'ENSG', etc.)
    df = df[~df.index.astype(str).str.startswith("__")]  # quita contadores especiales
    return df

def fetch_counts_from_url(url: str, timeout=(10, 600)) -> pd.DataFrame:
    data = _download_bytes(url, params=None, timeout=timeout)
    return _read_counts_from_bytes(data, url)

def fetch_geo_counts_df(gse_id: str, filename: Optional[str] = None) -> pd.DataFrame:
    """Descarga y lee el TSV(.gz) de counts a DataFrame (índice=GeneID) usando la página 'type=rnaseq_counts'."""
    if filename is None:
        filename = _find_rnaseq_counts_file(gse_id)
        if filename is None:
            raise FileNotFoundError(f"No encontré fichero de counts para {gse_id} (type=rnaseq_counts).")
    params = {"format": "file", "type": "rnaseq_counts", "acc": gse_id, "file": filename}
    data = _download_bytes(_GEO_DL, params=params)
    return _read_counts_from_bytes(data, filename)

def fetch_counts_any(gse_id: str, filename: Optional[str]) -> pd.DataFrame:
    """Si filename es URL completa → descarga y parsea esa URL. Si no, usa el flujo GEO estándar."""
    if _is_url(filename):
        return fetch_counts_from_url(filename)
    return fetch_geo_counts_df(gse_id, filename=filename)

# ======================= Detección de tipo de índice =======================
def _looks_like_ensembl(idx: pd.Index) -> bool:
    s = pd.Series(idx.astype(str))
    m = s.str.match(r"^ENSG\d+(?:\.\d+)?$")
    return bool((m.sum() / max(1, len(s))) > 0.6)

def _looks_like_symbol(idx: pd.Index) -> bool:
    s = pd.Series(idx.astype(str))
    m = s.str.match(r"^[A-Za-z][A-Za-z0-9\-\._]+$")
    return bool((m.sum() / max(1, len(s))) > 0.6)

def _annotation_index_is_ensembl(annot: pd.DataFrame) -> bool:
    return _looks_like_ensembl(annot.index)

# ======================= Anotación GEO (Human.GRCh38.p13) =======================
def fetch_geo_annotation_df(filename: str = "Human.GRCh38.p13.annot.tsv.gz") -> pd.DataFrame:
    """Descarga y lee la anotación (índice=GeneID o ENSG, con columna 'symbol') en memoria."""
    params = {"format": "file", "type": "rnaseq_counts", "file": filename}
    data = _download_bytes(_GEO_DL, params=params)
    compression = "gzip" if filename.lower().endswith(".gz") else None
    ann = pd.read_csv(io.BytesIO(data), sep="\t", header=0, low_memory=False,
                      compression=compression, dtype=str)

    # índice (GeneID/ENSG/etc.)
    id_col = next((c for c in ann.columns if c.lower() in {"geneid", "gene_id", "id"}), None)
    if id_col is None:
        # si no hay gene_id/geneid, prueba 'Gene'/'gene' o 'Ensembl'
        for c in ann.columns:
            if str(c).lower() in {"gene", "ensembl", "ensembl_id"}:
                id_col = c
                break
        if id_col is None:
            id_col = ann.columns[0]
    ann = ann.set_index(id_col)

    # columna símbolo
    cand = [c for c in ann.columns if re.search(r"(gene[_ ]?name|gene[_ ]?symbol|^symbol$)", c, flags=re.I)]
    if not cand:
        raise RuntimeError("No encuentro columna de símbolo en la anotación.")
    sym_col = cand[0]
    out = ann[[sym_col]].rename(columns={sym_col: "symbol"})
    out["symbol"] = out["symbol"].astype(str).str.strip()
    return out

# ======================= ENSG → SYMBOL (join o MyGene) =======================
def counts_to_symbol_with_annotation(counts: pd.DataFrame, annot: pd.DataFrame, how="sum") -> pd.DataFrame:
    """Join ENSG→SYMBOL cuando la anotación indexa por ENSG/GeneID compatible."""
    strip = lambda idx: pd.Index([re.sub(r"\.\d+$", "", str(x)) for x in idx], name="GeneID")
    counts2, annot2 = counts.copy(), annot.copy()
    counts2.index, annot2.index = strip(counts2.index), strip(annot2.index)
    merged = counts2.join(annot2[["symbol"]], how="left").dropna(subset=["symbol"])
    if how == "mean":
        agg = merged.groupby("symbol").mean(numeric_only=True)
    elif how == "median":
        agg = merged.groupby("symbol").median(numeric_only=True)
    else:
        agg = merged.groupby("symbol").sum(numeric_only=True)
    agg.index.name = "gene_symbol"
    return agg

def _ensg_to_symbol_mygene(ensg_ids: List[str], chunk_size: int = 300) -> pd.Series:
    """
    Fallback: mapea ENSG → SYMBOL vía MyGene.info usando POST (evita 414) y troceando.
    Devuelve una Series alineada al orden de ensg_ids (sin versión), con NaN si no hay match.
    """
    base = "https://mygene.info/v3/query"

    # limpiar versiones y deduplicar
    base_ids = [re.sub(r"\.\d+$", "", str(x)) for x in ensg_ids]
    uniq = sorted(set(base_ids))
    out_map: Dict[str, str] = {}

    # trocear para no exceder límites
    for i in range(0, len(uniq), chunk_size):
        chunk = uniq[i:i + chunk_size]

        # POST form-encoded evita URLs largas
        payload = {
            "q": ",".join(chunk),
            "scopes": "ensembl.gene",
            "fields": "symbol",
            "species": "human",
            "size": len(chunk),
        }
        r = requests.post(base, data=payload, timeout=60,
                          headers={"User-Agent": USER_AGENT})
        r.raise_for_status()

        data = r.json()
        # mygene puede devolver dict con 'hits' o lista directa
        hits = data.get("hits", data) if isinstance(data, dict) else data

        for hit in hits:
            q = hit.get("query")
            sym = hit.get("symbol")
            if q and sym:
                out_map[q] = sym

    # reconstruir en el mismo orden que ensg_ids (sin versión)
    base_ids_series = pd.Index([re.sub(r"\.\d+$", "", str(x)) for x in ensg_ids], name="GeneID")
    syms = [out_map.get(e, np.nan) for e in base_ids_series]
    return pd.Series(syms, index=base_ids_series, dtype="object")


def counts_to_symbol_if_needed(counts: pd.DataFrame,
                               annot_df: Optional[pd.DataFrame],
                               how: str = "sum",
                               force_map: bool = False) -> pd.DataFrame:
    """
    1) Ya símbolos → colapsa por duplicados.
    2) ENSG + anotación en ENSG → join directo.
    3) ENSG + anotación NO en ENSG → MyGene fallback.
    4) force_map (URL) + índice libre con ENSG embebido → extrae ENSG y aplica 2/3.
    5) Último recurso → colapsa por índice actual.
    """
    # 1) Ya son símbolos
    if _looks_like_symbol(counts.index) and not _looks_like_ensembl(counts.index):
        out = getattr(counts.groupby(counts.index), how)(numeric_only=True)
        out.index.name = "gene_symbol"
        return out

    # 2) Índice ENSG
    if _looks_like_ensembl(counts.index):
        if annot_df is not None and _annotation_index_is_ensembl(annot_df):
            return counts_to_symbol_with_annotation(counts, annot_df, how=how)
        # 3) anotación no ENSG -> MyGene
        sym_series = _ensg_to_symbol_mygene(list(counts.index))
        df = counts.copy()
        df.insert(0, "_symbol", sym_series.values)
        df = df.dropna(subset=["_symbol"])
        if df.empty:
            return df.drop(columns=["_symbol"])
        out = getattr(df.groupby("_symbol"), how)(numeric_only=True)
        out.index.name = "gene_symbol"
        return out

    # 4) force_map: índice libre con ENSG dentro (URLs)
    if annot_df is not None and force_map:
        new_idx, frac = _extract_ensembl_from_index(counts.index)
        if frac > 0.5:
            counts2 = counts.copy()
            counts2.index = new_idx
            counts2 = counts2.loc[counts2.index.notna()]
            if _annotation_index_is_ensembl(annot_df):
                return counts_to_symbol_with_annotation(counts2, annot_df, how=how)
            sym_series = _ensg_to_symbol_mygene(list(counts2.index))
            df = counts2.copy()
            df.insert(0, "_symbol", sym_series.values)
            df = df.dropna(subset=["_symbol"])
            if df.empty:
                return df.drop(columns=["_symbol"])
            out = getattr(df.groupby("_symbol"), how)(numeric_only=True)
            out.index.name = "gene_symbol"
            return out

    # 5) fallback: colapsa por índice tal cual
    out = getattr(counts.groupby(counts.index), how)(numeric_only=True)
    out.index.name = "gene_symbol"
    return out

# ======================= Metadata estilo pData (GEOparse en tmp) =======================
def _first_item(meta: dict, key: str, default=""):
    v = (meta or {}).get(key, default)
    if isinstance(v, (list, tuple)): return v[0] if v else default
    return v

def _norm_key(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[\s/\-]+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return s

def expand_characteristics_columns(pheno: pd.DataFrame) -> pd.DataFrame:
    ph = pheno.copy()
    char_cols = [c for c in ph.columns
                 if re.match(r"characteristics(_ch[12])?(\.\d+)?$", str(c), flags=re.I)
                 or str(c).lower().startswith("characteristics_ch")]
    extracted = defaultdict(list); all_keys = set()
    for idx, row in ph.iterrows():
        kv = {}
        for c in char_cols:
            val = row.get(c, None)
            if pd.isna(val) or str(val).strip() == "": continue
            lines = val if isinstance(val, (list, tuple)) else [val]
            flat = []
            for ln in lines:
                if isinstance(ln, str):
                    flat.extend([s for s in re.split(r"\s*;\s*|\s*\|\s*|\n+", ln) if s.strip() != ""])
            for item in flat:
                if ":" in item:
                    k, v = item.split(":", 1)
                    k = _norm_key(k); v = v.strip()
                    kv[k] = (kv.get(k, "") + (" | " if k in kv and kv[k] != v else "") + v).strip(" |")
                else:
                    kv["characteristics_free"] = (kv.get("characteristics_free","") + " | " + str(item).strip()).strip(" |")
        for k, v in kv.items():
            extracted[k].append((idx, v)); all_keys.add(k)
    if not all_keys: return ph
    add = pd.DataFrame(index=ph.index, columns=sorted(all_keys), dtype=object)
    for k, pairs in extracted.items():
        for idx, v in pairs: add.at[idx, k] = v
    for c in list(add.columns):
        if c in ph.columns: add.rename(columns={c: f"{c}_from_char"}, inplace=True)
    return pd.concat([ph, add], axis=1)

def get_gse_metadata(gse) -> pd.DataFrame:
    m = gse.metadata or {}
    row = {
        "gse_accession": gse.name,
        "title":          _first_item(m, "title", ""),
        "summary":        _first_item(m, "summary", ""),
        "overall_design": _first_item(m, "overall_design", ""),
        "type":           _first_item(m, "type", ""),
        "pubmed_id":      _first_item(m, "pubmed_id", ""),
        "submission_date":_first_item(m, "submission_date", ""),
        "status":         _first_item(m, "status", ""),
        "contributor":    ", ".join(m.get("contributor", [])) if "contributor" in m else "",
        "platforms":      ", ".join(sorted(gse.gpls.keys())),
        "n_samples":      len(gse.gsms),
    }
    return pd.DataFrame([row]).set_index("gse_accession")

def make_sample_metadata(gse) -> pd.DataFrame:
    rows = []
    for gsm_id, gsm in gse.gsms.items():
        meta = gsm.metadata or {}
        rows.append({
            "GSM": gsm_id,
            "title": _first_item(meta, "title", ""),
            "organism_ch1": _first_item(meta, "organism_ch1", _first_item(meta, "organism", "")),
            "source_name_ch1": _first_item(meta, "source_name_ch1", _first_item(meta, "source_name", "")),
            "platform_id": _first_item(meta, "platform_id", ""),
            "geo_accession": _first_item(meta, "geo_accession", gsm_id),
            "supplementary_file": "|".join(meta.get("supplementary_file", [])) if "supplementary_file" in meta else "",
        })
    base = pd.DataFrame(rows).set_index("GSM")
    pheno = gse.phenotype_data.copy(); pheno.index.name = "GSM"
    merged = base.join(pheno, how="outer", lsuffix="_gsm", rsuffix="_pheno")

    for col in ["title","organism_ch1","source_name_ch1","platform_id","geo_accession","supplementary_file"]:
        cg, cp = f"{col}_gsm", f"{col}_pheno"
        if cg in merged.columns and cp in merged.columns:
            merged[col] = merged[cg].combine_first(merged[cp]); merged.drop(columns=[cg, cp], inplace=True)
        elif cg in merged.columns:
            merged.rename(columns={cg: col}, inplace=True)
        elif cp in merged.columns:
            merged.rename(columns={cp: col}, inplace=True)

    merged_expanded = expand_characteristics_columns(merged)
    front = [c for c in ["title","platform_id","organism_ch1","source_name_ch1","supplementary_file"]
             if c in merged_expanded.columns]
    resto = [c for c in merged_expanded.columns if c not in front]
    cols = front + resto
    return merged_expanded.loc[:, cols]

def get_geo_metadata_cached(gse_id: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    geo_dir = CACHE_DIR / "geoparse"
    geo_dir.mkdir(parents=True, exist_ok=True)

    # GEOparse reusa si el fichero ya existe en destdir
    gse = gp.get_GEO(geo=gse_id, destdir=str(geo_dir), annotate_gpl=False, silent=True)

    gse_meta_df = get_gse_metadata(gse)
    gsm_meta_df = make_sample_metadata(gse)
    return gse_meta_df, gsm_meta_df

# ======================= Loader de un GSE (todo en memoria) =======================
def load_geo_rnaseq_with_metadata_mem(
    gse_id: str,
    counts_filename: Optional[str] = None,              # puede ser nombre o URL
    annot_filename: Optional[str] = "Human.GRCh38.p13.annot.tsv.gz",
    collapse: str = "sum",
):
    # 1) counts (GEO o URL)
    counts_df = fetch_counts_any(gse_id, counts_filename)

    # 2) metadata
    gse_meta_df, gsm_meta_df = get_geo_metadata_cached(gse_id)

    # 3) anotación y símbolo
    annot_df = None
    if annot_filename:
        try:
            annot_df = fetch_geo_annotation_df(annot_filename)
        except Exception as e:
            print(f"Aviso anotación: {e}")

    # fuerza mapeo si vino por URL (http/ftp)
    force_map = bool(counts_filename and _is_url(counts_filename))

    counts_symbol = counts_to_symbol_if_needed(
        counts_df, annot_df, how=collapse, force_map=force_map
    )

    return {
        "counts": counts_df,
        "counts_symbol": counts_symbol,
        "gse_meta": gse_meta_df,
        "gsm_meta": gsm_meta_df,
        "annot": annot_df,
    }

# ======================= Batch (varios GSE → memoria) =======================
def batch_load_geo_rnaseq_mem(
    jobs: Dict[str, Optional[str]] | Iterable[str],
    annot_filename: Optional[str] = "Human.GRCh38.p13.annot.tsv.gz",
    collapse: str = "sum",
):
    """
    jobs puede ser:
      - lista de GSE: ["GSE141945","GSE206838", ...]  (autodetecta filename)
      - dict {GSE: counts_filename_o_URL_o_None}
    Devuelve: {"results": {GSE: {...}}, "series_meta": DF, "sample_meta": DF}
    """
    if not isinstance(jobs, dict):
        jobs = {gse: None for gse in jobs}

    results = {}
    series_meta_rows, sample_meta_rows = [], []

    for gse_id, fname in jobs.items():
        try:
            res = load_geo_rnaseq_with_metadata_mem(
                gse_id,
                counts_filename=fname,
                annot_filename=annot_filename,
                collapse=collapse,
            )
            results[gse_id] = res
            series_meta_rows.append(res["gse_meta"])
            gm = res["gsm_meta"].copy()
            gm.insert(0, "gse_accession", gse_id)
            sample_meta_rows.append(gm)
            print(f"[OK] {gse_id} ✅")
        except Exception as e:
            print(f"❌[ERROR] {gse_id}: {e}❌")

    series_meta = pd.concat(series_meta_rows) if series_meta_rows else pd.DataFrame()
    sample_meta = pd.concat(sample_meta_rows) if sample_meta_rows else pd.DataFrame()
    return {"results": results, "series_meta": series_meta, "sample_meta": sample_meta}


# ☄️ Descarga RNASEQ

In [2]:

# ======================= EJEMPLO DE USO (dinámicas como RNASEQ) =======================
rnaseq_jobs = {
    #"GSE164458": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE164nnn/GSE164458/suppl/GSE164458_BrighTNess_RNAseq_log2_Processed_ASTOR.txt.gz",
    "GSE96058": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE96nnn/GSE96058/suppl/GSE96058_gene_expression_3273_samples_and_136_replicates_transformed.csv.gz",
 
}


batch_rnaseq = batch_load_geo_rnaseq_mem(
    rnaseq_jobs,
    annot_filename="Human.GRCh38.p13.annot.tsv.gz",
    collapse="sum"
)

# variables dinámicas por GSE
for gse_id, res in batch_rnaseq["results"].items():
    globals()[f"counts_{gse_id}"]        = res["counts"]
    globals()[f"counts_symbol_{gse_id}"] = res["counts_symbol"]
    globals()[f"gse_meta_{gse_id}"]      = res["gse_meta"]
    globals()[f"{gse_id}_meta"]          = res["gsm_meta"]
    if res.get("annot") is not None:
        globals()[f"annot_{gse_id}"] = res["annot"]
    print(f"creado counts_{gse_id}, counts_symbol_{gse_id}, gse_meta_{gse_id} y {gse_id}_meta")


[OK] GSE96058 ✅
creado counts_GSE96058, counts_symbol_GSE96058, gse_meta_GSE96058 y GSE96058_meta


## GSE96058 🎉 

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
subtype_map = {
    'LumA': 'LumA',
    'LumB': 'LumB',
    'Her2': 'HER2',
    'Basal': 'TNBC',
    'Normal': 'Normal'
}

GSE96058_meta['characteristics_ch1.20.pam50 subtype'] = GSE96058_meta['characteristics_ch1.20.pam50 subtype'].map(subtype_map)
counts_symbol_GSE96058 = counts_symbol_GSE96058[GSE96058_meta['title']]


In [5]:
GSE96058_meta.loc[:, 'label'] = GSE96058_meta['characteristics_ch1.20.pam50 subtype']
GSE96058_meta['batch'] = 'GSE96058'
GSE96058_meta['sample'] = GSE96058_meta['title'].copy()

In [6]:
GSE96058_meta = GSE96058_meta[['label','batch','sample']]

In [7]:
print(counts_symbol_GSE96058.shape[1], GSE96058_meta.shape[0])

3409 3409


In [33]:
GSE96058_meta.to_csv('./Data/GSE96058_meta.csv')
counts_symbol_GSE96058.to_csv('./Data/counts_symbol_GSE96058.csv')

## GSE164458 🎉  


# ☄️ Leemos TCGA_BCRA


In [8]:
# -*- coding: utf-8 -*-
"""
Descarga automática (UCSC Xena) de TCGA-BRCA:
- Matriz de expresión (RNA-seq)
- Matriz fenotípica/clinical
- Metadata con clasificación: HER2, LumA, LumB, TNBC y Normal

Salida:
  ./tcga_brca_auto/
    - expression_samples_x_genes.tsv.gz
    - expression_genes_x_samples.tsv.gz
    - metadata_tcga_brca_5class.tsv
    - download_manifest.json
"""

from __future__ import annotations
import io
import json
import gzip
import re
from pathlib import Path
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import requests

XENA_HOST = "https://tcga.xenahubs.net"
OUTDIR = Path("tcga_brca_auto")
OUTDIR.mkdir(parents=True, exist_ok=True)

TCGA_SAMPLE16_RE = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-\d{2}[A-Z]?)", re.I)
TCGA_PATIENT_RE  = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", re.I)

# Intentamos varias opciones por robustez (según hub/versiones)
EXPR_DATASET_CANDIDATES = [
    "TCGA.BRCA.sampleMap/HiSeqV2",
    "TCGA.BRCA.sampleMap/HiSeqV2_PANCAN",
]
PHENO_DATASET_CANDIDATES = [
    "TCGA.BRCA.sampleMap/BRCA_clinicalMatrix",
]

def _http_get(url: str, timeout: int = 180) -> bytes:
    r = requests.get(url, stream=True, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code} al descargar {url}")
    return r.content

def download_xena_dataset(dataset_id_candidates: List[str], out_prefix: str) -> Tuple[Path, str, str]:
    """
    Devuelve: (path_local_tsv_gz, dataset_id_usado, url_usada)
    """
    out_path = OUTDIR / f"{out_prefix}.tsv.gz"
    if out_path.exists():
        return out_path, "cached", "cached"

    last_error = None
    for ds in dataset_id_candidates:
        for url in (f"{XENA_HOST}/download/{ds}", f"{XENA_HOST}/download/{ds}.gz"):
            try:
                raw = _http_get(url)
                # Xena a veces sirve gzip aunque la URL no termine en .gz
                if raw[:2] == b"\x1f\x8b":
                    try:
                        with gzip.GzipFile(fileobj=io.BytesIO(raw)) as gz:
                            txt = gz.read()
                    except OSError:
                        txt = raw
                else:
                    txt = raw

                # Validación mínima: TSV
                if b"\t" not in txt[:1000]:
                    raise RuntimeError("Contenido descargado no parece TSV")

                # Guardamos SIEMPRE comprimido
                with gzip.open(out_path, "wb") as f:
                    f.write(txt)
                return out_path, ds, url
            except Exception as e:
                last_error = e
                continue

    raise RuntimeError(f"No se pudo descargar ningún dataset de {dataset_id_candidates}. Último error: {last_error}")

def read_tsv_gz(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", low_memory=False)

def _looks_like_tcga_barcodes(values, n=30) -> bool:
    vals = [str(v) for v in list(values)[:n]]
    hits = 0
    for v in vals:
        if TCGA_SAMPLE16_RE.search(v) or TCGA_PATIENT_RE.search(v):
            hits += 1
    return hits >= max(1, len(vals)//4)

def load_expression_auto(expr_path: Path) -> pd.DataFrame:
    """
    Devuelve matriz samples x genes (índice=sample).
    """
    df = read_tsv_gz(expr_path)
    if df.empty:
        raise ValueError("La matriz de expresión está vacía")

    first_col = str(df.columns[0])

    # Caso A: columnas = samples, filas = genes/probes (común en Xena)
    if _looks_like_tcga_barcodes(df.columns[1:]):
        expr = df.set_index(first_col).copy()
        expr = expr.apply(pd.to_numeric, errors="coerce")
        expr = expr.T
        expr.index.name = "sample"
        expr.columns = expr.columns.astype(str)
        return expr

    # Caso B: filas = samples, columnas = genes
    if _looks_like_tcga_barcodes(df[first_col].astype(str).values):
        expr = df.set_index(first_col).copy()
        expr = expr.apply(pd.to_numeric, errors="coerce")
        expr.index.name = "sample"
        expr.columns = expr.columns.astype(str)
        return expr

    raise ValueError("No pude inferir orientación de la matriz de expresión.")

def _find_col(df: pd.DataFrame, patterns: List[str], required: bool=False) -> Optional[str]:
    cols = [str(c) for c in df.columns]
    for pat in patterns:
        rx = re.compile(pat, flags=re.I)
        hits = [c for c in cols if rx.search(c)]
        if hits:
            return hits[0]
    if required:
        raise ValueError(f"No encontré una columna con patrones {patterns}")
    return None

def _extract_sample16(x) -> Optional[str]:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    m = TCGA_SAMPLE16_RE.search(s)
    if m:
        return m.group(1).upper()
    m2 = TCGA_PATIENT_RE.search(s)  # fallback a paciente
    if m2:
        return m2.group(1).upper()
    return s

def _extract_patient(x) -> Optional[str]:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    m = TCGA_PATIENT_RE.search(s)
    return m.group(1).upper() if m else np.nan

def _is_pos(v) -> bool:
    if pd.isna(v):
        return False
    s = str(v).strip().lower()
    if s in {"", "nan", "na", "not reported", "not available", "none"}:
        return False
    return bool(re.search(r"\b(pos|positive|amplified|3\+|1)\b", s))

def _is_neg(v) -> bool:
    if pd.isna(v):
        return False
    s = str(v).strip().lower()
    if s in {"", "nan", "na", "not reported", "not available", "none"}:
        return False
    return bool(re.search(r"\b(neg|negative|not amplified|0|1\+)\b", s))

def _norm_pam50(v) -> Optional[str]:
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    s2 = s.replace("_", " ").replace("-", " ")
    if "luma" in s2 or "luminal a" in s2:
        return "LumA"
    if "lumb" in s2 or "luminal b" in s2:
        return "LumB"
    if "her2" in s2:
        return "HER2"
    if "basal" in s2:
        return "Basal"
    if "normal" in s2:
        return "Normal-like"
    return np.nan

def build_metadata(expr: pd.DataFrame, pheno: pd.DataFrame) -> pd.DataFrame:
    # ===== Base desde expresión =====
    meta = pd.DataFrame({"sample": expr.index.astype(str).str.upper()})
    meta["sample16"] = meta["sample"].str.extract(TCGA_SAMPLE16_RE, expand=False).str.upper()
    meta["patient"]  = meta["sample"].str.extract(TCGA_PATIENT_RE,  expand=False).str.upper()
    meta["sample_code"] = meta["sample16"].str.slice(13, 15)
    meta["tcga_sample_type_from_barcode"] = np.select(
        [
            meta["sample_code"].eq("11"),
            meta["sample_code"].isin([f"{i:02d}" for i in range(1, 10)]),
        ],
        ["Solid Tissue Normal", "Tumor"],
        default="Other"
    )

    # ===== Fenotipo clínico (Xena) =====
    ph = pheno.copy()
    ph.columns = [str(c) for c in ph.columns]

    sample_col = _find_col(ph, [r"^sample(_?id)?$", r"sample", r"barcode", r"submitter"], required=True)
    ph["sample_raw"] = ph[sample_col].astype(str)
    ph["sample16"] = ph["sample_raw"].map(_extract_sample16)
    ph["patient"] = ph["sample_raw"].map(_extract_patient)

    # Columnas candidatas (tolerante)
    pam_col = _find_col(ph, [r"pam50call_rnaseq", r"subtype.*pam50", r"pam50"], required=False)
    sample_type_col = _find_col(ph, [r"^sample_type$", r"sample type"], required=False)

    er_col = _find_col(ph, [
        r"breast_carcinoma_estrogen_receptor_status",
        r"er_status_by_ihc",
        r"\ber\b.*status",
        r"estrogen.*receptor.*status",
    ], required=False)

    pr_col = _find_col(ph, [
        r"breast_carcinoma_progesterone_receptor_status",
        r"pr_status_by_ihc",
        r"\bpr\b.*status",
        r"progesterone.*receptor.*status",
    ], required=False)

    her2_col = _find_col(ph, [
        r"lab_proc_her2.*receptor_status",
        r"her2_status_by_ihc",
        r"her2_final_status",
        r"her2.*status",
    ], required=False)

    keep = ["sample16", "patient"]
    for c in [sample_type_col, pam_col, er_col, pr_col, her2_col]:
        if c and c not in keep:
            keep.append(c)
    phs = ph[keep].copy()

    # Normalizaciones
    if pam_col:
        phs["pam50_raw"] = phs[pam_col]
        phs["pam50"] = phs[pam_col].map(_norm_pam50)
    else:
        phs["pam50_raw"] = np.nan
        phs["pam50"] = np.nan

    if sample_type_col:
        phs["xena_sample_type"] = phs[sample_type_col].astype(str)
    else:
        phs["xena_sample_type"] = np.nan

    for outcol, src in [("ER_status_raw", er_col), ("PR_status_raw", pr_col), ("HER2_status_raw", her2_col)]:
        phs[outcol] = phs[src] if src else np.nan

    phs["ER_pos"] = phs["ER_status_raw"].map(_is_pos)
    phs["PR_pos"] = phs["PR_status_raw"].map(_is_pos)
    phs["HER2_pos"] = phs["HER2_status_raw"].map(_is_pos)

    phs["ER_neg"] = phs["ER_status_raw"].map(_is_neg)
    phs["PR_neg"] = phs["PR_status_raw"].map(_is_neg)
    phs["HER2_neg"] = phs["HER2_status_raw"].map(_is_neg)

    phs["TNBC_from_receptors"] = phs["ER_neg"] & phs["PR_neg"] & phs["HER2_neg"]

    # Deduplicamos priorizando filas con más info
    phs["_score"] = (
        phs["pam50"].notna().astype(int) * 4
        + phs[["ER_status_raw", "PR_status_raw", "HER2_status_raw"]].notna().sum(axis=1)
    )
    ph_by_sample = phs.dropna(subset=["sample16"]).sort_values("_score", ascending=False).drop_duplicates("sample16")
    ph_by_patient = phs.dropna(subset=["patient"]).sort_values("_score", ascending=False).drop_duplicates("patient")

    # Merge sample-level
    out = meta.merge(
        ph_by_sample.drop(columns=["patient"], errors="ignore"),
        on="sample16",
        how="left",
        suffixes=("", "_dup")
    )

    # Fallback patient-level
    out = out.merge(
        ph_by_patient.add_suffix("_pt"),
        left_on="patient",
        right_on="patient_pt",
        how="left"
    )

    fill_cols = [
        "pam50_raw", "pam50", "xena_sample_type",
        "ER_status_raw", "PR_status_raw", "HER2_status_raw",
        "ER_pos", "PR_pos", "HER2_pos",
        "ER_neg", "PR_neg", "HER2_neg",
        "TNBC_from_receptors"
    ]
    for c in fill_cols:
        c_pt = f"{c}_pt"
        if c not in out.columns and c_pt in out.columns:
            out[c] = out[c_pt]
        elif c in out.columns and c_pt in out.columns:
            out[c] = out[c].where(out[c].notna(), out[c_pt])

    # ===== Clasificación pedida =====
    xena_st = out.get("xena_sample_type", pd.Series(index=out.index, dtype="object")).astype(str).str.lower()
    is_normal = out["sample_code"].eq("11") | xena_st.str.contains("normal", na=False)
    is_tumor  = out["sample_code"].isin([f"{i:02d}" for i in range(1, 10)]) | xena_st.str.contains("tumor", na=False)

    out["label_5class"] = pd.Series([pd.NA] * len(out), dtype="object")
    out.loc[is_normal, "label_5class"] = "Normal"

    # TNBC por receptores (prioridad)
    out.loc[(is_tumor) & (out["TNBC_from_receptors"] == True), "label_5class"] = "TNBC"

    # PAM50 para LumA/LumB/HER2
    free = out["label_5class"].isna()
    out.loc[free & (out["pam50"] == "LumA"), "label_5class"] = "LumA"
    out.loc[free & (out["pam50"] == "LumB"), "label_5class"] = "LumB"
    out.loc[free & (out["pam50"] == "HER2"), "label_5class"] = "HER2"

    # Basal-like (si quieres inspeccionarlo aparte)
    out["basal_like_pam50"] = out["pam50"].eq("Basal")

    out["use_for_requested_5class"] = out["label_5class"].isin(["HER2", "LumA", "LumB", "TNBC", "Normal"])

    # orden columnas
    preferred_cols = [
        "sample", "sample16", "patient",
        "sample_code", "tcga_sample_type_from_barcode", "xena_sample_type",
        "pam50_raw", "pam50",
        "ER_status_raw", "PR_status_raw", "HER2_status_raw",
        "ER_pos", "PR_pos", "HER2_pos", "ER_neg", "PR_neg", "HER2_neg",
        "TNBC_from_receptors", "basal_like_pam50",
        "label_5class", "use_for_requested_5class"
    ]
    existing = [c for c in preferred_cols if c in out.columns]
    others = [c for c in out.columns if c not in existing and not c.endswith("_pt") and c not in {"patient_pt", "_score"}]
    out = out[existing + others]

    return out

def main():
    print(">> Descargando expresión...")
    expr_path, expr_ds, expr_url = download_xena_dataset(EXPR_DATASET_CANDIDATES, out_prefix="xena_brca_expression_raw")

    print(">> Descargando fenotipo/clinical...")
    pheno_path, ph_ds, ph_url = download_xena_dataset(PHENO_DATASET_CANDIDATES, out_prefix="xena_brca_clinical_raw")

    print(">> Leyendo y transformando expresión...")
    expr = load_expression_auto(expr_path)
    expr = expr.loc[~expr.index.duplicated(keep="first")]
    expr = expr.loc[:, ~expr.columns.duplicated(keep="first")]

    print(">> Leyendo fenotipo...")
    pheno = read_tsv_gz(pheno_path)

    print(">> Construyendo metadata...")
    meta = build_metadata(expr, pheno)

    # Guardar outputs
    expr_sxg_path = OUTDIR / "expression_samples_x_genes.tsv.gz"
    expr_gxs_path = OUTDIR / "expression_genes_x_samples.tsv.gz"
    meta_path = OUTDIR / "metadata_tcga_brca_5class.tsv"
    manifest_path = OUTDIR / "download_manifest.json"

    expr.to_csv(expr_sxg_path, sep="\t", compression="gzip")
    expr.T.to_csv(expr_gxs_path, sep="\t", compression="gzip")
    meta.to_csv(meta_path, sep="\t", index=False)

     # Conteos serializables a JSON (evita pd.NA como clave)
    counts = (
        meta["label_5class"]
        .astype("object")
        .where(meta["label_5class"].notna(), "NA")
        .value_counts(dropna=False)
    )
    class_counts_json = {str(k): int(v) for k, v in counts.items()}
    
    manifest = {
        "xena_host": XENA_HOST,
        "expression_dataset_used": str(expr_ds),
        "expression_url_used": str(expr_url),
        "phenotype_dataset_used": str(ph_ds),
        "phenotype_url_used": str(ph_url),
        "n_samples_expression": int(expr.shape[0]),
        "n_genes_expression": int(expr.shape[1]),
        "n_samples_metadata": int(meta.shape[0]),
        "class_counts_label_5class": class_counts_json,
        "notes": {
            "TNBC_definition": "ER-, PR-, HER2- (receptor fields in clinical matrix)",
            "Normal_definition": "barcode sample code 11 and/or clinical sample_type normal",
            "LumA_LumB_HER2_definition": "PAM50 column auto-detected (e.g., PAM50Call_RNAseq)"
        }
    }
    
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print("\n=== LISTO ===")
    print(f"Expresión (samples x genes): {expr_sxg_path}")
    print(f"Expresión (genes x samples): {expr_gxs_path}")
    print(f"Metadata: {meta_path}")
    print(f"Manifest: {manifest_path}")
    print("\nConteos label_5class:")
    print(meta["label_5class"].value_counts(dropna=False))

    n_na = int(meta["label_5class"].isna().sum())
    if n_na > 0:
        print(f"\n[INFO] {n_na} muestras quedaron sin clase en 'label_5class' (faltantes clínicos u otros casos).")
        print("       Puedes filtrar con: meta['use_for_requested_5class'] == True")
    return expr, meta, pheno

expr, meta, pheno = main()

>> Descargando expresión...
>> Descargando fenotipo/clinical...
>> Leyendo y transformando expresión...
>> Leyendo fenotipo...
>> Construyendo metadata...

=== LISTO ===
Expresión (samples x genes): tcga_brca_auto/expression_samples_x_genes.tsv.gz
Expresión (genes x samples): tcga_brca_auto/expression_genes_x_samples.tsv.gz
Metadata: tcga_brca_auto/metadata_tcga_brca_5class.tsv
Manifest: tcga_brca_auto/download_manifest.json

Conteos label_5class:
label_5class
LumA      421
<NA>      314
LumB      193
TNBC      117
Normal    114
HER2       59
Name: count, dtype: int64

[INFO] 314 muestras quedaron sin clase en 'label_5class' (faltantes clínicos u otros casos).
       Puedes filtrar con: meta['use_for_requested_5class'] == True


In [9]:
TCGA_meta = meta[meta["use_for_requested_5class"] == True].copy()
counts_symbol_TCGA = expr.loc[TCGA_meta["sample"]].copy()
counts_symbol_TCGA = counts_symbol_TCGA.T

In [10]:
TCGA_meta['label'] = TCGA_meta['label_5class']
TCGA_meta['batch'] = 'TCGA'
TCGA_meta = TCGA_meta[['label','batch','sample']]

In [11]:
TCGA_meta.index = TCGA_meta['sample']
TCGA_meta

,label,batch,sample
sample,,,
TCGA-AR-A5QQ-01,TNBC,TCGA,TCGA-AR-A5QQ-01
TCGA-D8-A1JA-01,HER2,TCGA,TCGA-D8-A1JA-01
TCGA-BH-A0BQ-01,LumA,TCGA,TCGA-BH-A0BQ-01
TCGA-BH-A0BT-01,LumA,TCGA,TCGA-BH-A0BT-01
TCGA-A8-A06X-01,LumB,TCGA,TCGA-A8-A06X-01
...,...,...,...
TCGA-E9-A5FL-01,TNBC,TCGA,TCGA-E9-A5FL-01
TCGA-AC-A2FB-11,Normal,TCGA,TCGA-AC-A2FB-11
TCGA-A2-A3XT-01,TNBC,TCGA,TCGA-A2-A3XT-01


In [34]:
TCGA_meta.to_csv('./Data/TCGA_meta.csv')
counts_symbol_TCGA.to_csv('./Data/counts_symbol_TCGA.csv')

In [13]:
del expr
del meta


#  🌍 Juntar todos los RNSASEQ y todo metadata RNASEQ


In [14]:
# ================================================
#  Construir mega-tablas separadas: COUNTS vs NORMALIZADOS
#  - Busca variables cuyo nombre empiece por expr_prefix (p.ej., "counts_symbol_")
#  - Clasifica cada estudio con una heurística (counts vs normalizado)
#  - Alinea genes por grupo y concatena
#  - Devuelve:
#       expr_counts_mega, meta_counts_mega,
#       expr_norm_mega,   meta_norm_mega,
#       scale_report
# ================================================
# ================================================
#  Mega-tablas separadas COUNTS vs NORMALIZADOS
#  (selector de metadata robusto + separación por escala)
# ================================================
import re, os
import numpy as np
import pandas as pd

PREFER_INTERSECTION = True
MIN_COMMON_GENES_WARN = 1000

# ----------------- Helpers -----------------
def _norm_sample_id(x: str) -> str:
    s = os.path.basename(str(x))
    s = re.sub(r"\.CEL(\.gz)?$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\.(fastq|fq|bam|sam|cram)(\.gz)?$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\.\d+$", "", s)  # quita .1, .2 finales
    return s.strip()

def _candidate_sample_columns():
    return [
        "sample","assay name","sample name","sample_name","source name",
        "array data file","derived array data file","geo_accession","array data matrix file",
        "run","run accession","run_accession","srr","experiment","experiment accession",
        "biosample","biosample accession","sample accession","sample_accession","title",
        "sample_barcode","aliquot_barcode","submitter_id","case_id",
        "name","id","file_name","file","specimen","specimen_barcode",
    ]

def _pick_best_meta(study_id: str, globs: dict, expr_cols) -> tuple[str|None, pd.DataFrame|None, float]:
    expr_norm = {_norm_sample_id(c) for c in map(str, expr_cols)}
    names_in_priority = (f"{study_id}_meta", f"gse_meta_{study_id}")
    best = None  # (score, overlap, name, meta_df_con_sample)

    for nm in names_in_priority:
        if nm not in globs:
            continue
        mdf = globs[nm]
        if not isinstance(mdf, pd.DataFrame):
            mdf = pd.DataFrame(mdf)
        mdf = mdf.copy()
        mdf.columns = [str(c).strip() for c in mdf.columns]
        cols_lower = {c.lower(): c for c in mdf.columns}

        if "sample" in cols_lower:
            series = mdf[cols_lower["sample"]].astype(str).map(_norm_sample_id)
            overlap = len(expr_norm & set(series)) / max(1, len(expr_norm))
            score = 10.0 + overlap
            meta_df = mdf.copy()
            meta_df["sample"] = series.values
            cand = (score, overlap, nm, meta_df)
        else:
            best_col, best_overlap = None, -1.0
            for key in _candidate_sample_columns():
                if key.lower() in cols_lower:
                    col = cols_lower[key.lower()]
                    vals = mdf[col].astype(str).map(_norm_sample_id)
                    ovlp = len(expr_norm & set(vals)) / max(1, len(expr_norm))
                    if ovlp > best_overlap:
                        best_overlap, best_col = ovlp, col
            if best_col is not None:
                score = 5.0 + best_overlap
                tmp = mdf.copy()
                tmp["sample"] = tmp[best_col].astype(str).map(_norm_sample_id)
                cand = (score, best_overlap, nm, tmp)
            else:
                cand = (0.0, 0.0, nm, None)

        if best is None or cand[0] > best[0]:
            best = cand

    if best is None or best[3] is None:
        return None, None, 0.0
    return best[2], best[3], float(best[1])

def _classify_expr_scale(X: pd.DataFrame):
    Xn = X.select_dtypes(include=[np.number])
    V = Xn.to_numpy()
    if V.size == 0:
        return {"group":"unknown","reason":"sin columnas numéricas",
                "nonneg":None,"frac_integers":None,"cv":None,"max":None,"q95":None,"zero_frac":None}

    nonneg = bool((V >= 0).all())
    frac_int = float(np.mean(np.isclose(V, np.rint(V), rtol=0, atol=1e-9)))
    col_sums = V.sum(axis=0)
    sum_mean = float(np.mean(col_sums)); sum_std = float(np.std(col_sums))
    cv = float(sum_std / (sum_mean + 1e-12))
    q50 = float(np.nanmedian(V)); q95 = float(np.nanpercentile(V, 95)); vmax = float(np.nanmax(V))
    zero_frac = float(np.mean(V == 0))
    around_1e6 = bool(np.all(np.isclose(col_sums, 1_000_000, rtol=0.15, atol=0)))
    looks_log  = (vmax <= 30 and q95 <= 20) or (q50 <= 2 and vmax <= 50)

    if not nonneg:
        group, reason = "normalized", "hay valores negativos (típico VST/rlog)"
    elif looks_log and frac_int < 0.98:
        group, reason = "normalized", "rango pequeño y decimales (log/transformado)"
    elif frac_int >= 0.98 and vmax >= 50 and cv >= 0.10:
        group, reason = "counts", "enteros, sumas por muestra variables (CV alto)"
    elif around_1e6:
        group, reason = "normalized", "sumas por muestra ≈ 1e6 (CPM/TPM)"
    elif frac_int < 0.98 and cv < 0.05:
        group, reason = "normalized", "decimales y sumas ~constantes (size-factor)"
    elif frac_int < 0.98:
        group, reason = "normalized", "decimales y sumas variables"
    else:
        group, reason = "unknown", "reglas no concluyentes"

    return {
        "group": group, "reason": reason,
        "nonneg": nonneg, "frac_integers": frac_int, "cv": cv,
        "max": vmax, "q95": q95, "zero_frac": zero_frac
    }

def _dedup_index(df: pd.DataFrame, how: str = "first") -> pd.DataFrame:
    """
    Asegura índice único por filas (genes). how: 'first'|'sum'|'mean'|'median'|'max'
    """
    if not df.index.has_duplicates:
        return df
    if how == "first":
        return df[~df.index.duplicated(keep="first")]
    if how in ("sum","mean","median","max"):
        agg = getattr(pd.core.groupby.generic.DataFrameGroupBy, how, None)
        # agrupamos por índice
        if how == "sum":
            return df.groupby(level=0).sum(numeric_only=True)
        if how == "mean":
            return df.groupby(level=0).mean(numeric_only=True)
        if how == "median":
            return df.groupby(level=0).median(numeric_only=True)
        if how == "max":
            return df.groupby(level=0).max(numeric_only=True)
    # fallback
    return df[~df.index.duplicated(keep="first")]

def _make_unique_columns(cols):
    seen = {}
    out = []
    for c in cols:
        c = str(c)
        if c not in seen:
            seen[c] = 1
            out.append(c)
        else:
            k = seen[c]; seen[c] += 1
            out.append(f"{c}__dup{k}")
    return out

# ----------------- Principal -----------------
def build_split_mega_tables_from_globals(
    globs: dict,
    expr_prefix: str = "counts_symbol_",
    prefer_intersection: bool | None = None,
    min_common_genes_warn: int | None = None,
    unknown_to: str = "normalized"  # dónde enviar 'unknown': 'normalized' o 'counts'
):
    if prefer_intersection is None:
        prefer_intersection = globals().get("PREFER_INTERSECTION", True)
    if min_common_genes_warn is None:
        min_common_genes_warn = globals().get("MIN_COMMON_GENES_WARN", 1000)

    expr_varnames = [nm for nm in globs.keys() if nm.startswith(expr_prefix)]
    if not expr_varnames:
        raise RuntimeError(f"No se encontraron objetos que empiecen por '{expr_prefix}'.")

    study_ids = [nm[len(expr_prefix):] for nm in expr_varnames]
    expr_pairs = sorted(zip(study_ids, expr_varnames), key=lambda x: x[0])
    print("Detectados estudios:", ", ".join(study_ids))

    expr_counts_list, meta_counts_list = [], []
    expr_norm_list,   meta_norm_list   = [], []
    report_rows = []

    for study_id, varname in expr_pairs:
        X = globs[varname]
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"{varname} no es un DataFrame.")
        if X.index.name is None:
            X.index.name = "gene"

        orig_cols = [str(c) for c in X.columns]

        # Clasificación de escala
        scale = _classify_expr_scale(X)
        group = scale["group"]
        if group == "unknown":
            print(f" · {study_id}: escala 'unknown' → '{unknown_to}' ({scale['reason']})")
            group = unknown_to
        else:
            print(f" · {study_id}: {group} ({scale['reason']})")

        # Selector de metadata
        meta_name, meta_df, overlap = _pick_best_meta(study_id, globs, orig_cols)
        if meta_df is None:
            print(f"   (aviso) {study_id}: sin metadata utilizable → uso columnas de expresión")
            meta_df = pd.DataFrame({"sample": orig_cols})
            meta_name = None
            overlap = 1.0

        # Prefijar columnas y normalizar IDs
        prefixed_cols = [f"{study_id}:{_norm_sample_id(c)}" for c in orig_cols]
        X_pref = X.copy()
        X_pref.columns = _make_unique_columns(prefixed_cols)  # por si hay columnas duplicadas

        # --- NUEVO: deduplicar genes por índice según el grupo ---
        if X_pref.index.has_duplicates:
            dups = int(X_pref.index.duplicated().sum())
            if group == "counts":
                print(f"   (dedup) {study_id}: {dups} genes duplicados → agrego por SUM")
                X_pref = _dedup_index(X_pref, how="sum")
            else:
                print(f"   (dedup) {study_id}: {dups} genes duplicados → agrego por MEAN")
                X_pref = _dedup_index(X_pref, how="mean")

        # Normalizar sample en metadata y prefijar
        meta_df = meta_df.copy()
        meta_df["sample"] = meta_df["sample"].astype(str).map(_norm_sample_id).map(lambda s: f"{study_id}:{s}")
        meta_df["batch"]  = study_id

        # Añadir al grupo
        if group == "counts":
            expr_counts_list.append(X_pref); meta_counts_list.append(meta_df)
        else:
            expr_norm_list.append(X_pref);   meta_norm_list.append(meta_df)

        report_rows.append({
            "study_id": study_id,
            "varname": varname,
            "meta_used": meta_name if meta_name is not None else "from_expr_cols",
            "meta_overlap": round(float(overlap), 3),
            "group": group, "reason": scale["reason"],
            "nonneg": scale["nonneg"], "frac_integers": scale["frac_integers"],
            "sum_cv": scale["cv"], "q95": scale["q95"], "max": scale["max"],
            "zero_frac": scale["zero_frac"],
        })

    scale_report = pd.DataFrame(report_rows).sort_values(["group","study_id"]).reset_index(drop=True)

    def _align_concat(expr_list, meta_list, group_name: str):
        if not expr_list:
            print(f" · No hay estudios en grupo '{group_name}'.")
            return None, None

        # Alinear genes (con índices ya únicos en cada df)
        gene_sets = [set(df.index) for df in expr_list]
        if prefer_intersection:
            common_genes = sorted(set.intersection(*gene_sets))
            if len(common_genes) < min_common_genes_warn:
                print(f" ⚠️ [{group_name}] genes comunes = {len(common_genes)} (revisa anotaciones/platforms).")
            expr_list_aligned = [df.loc[common_genes] for df in expr_list]
        else:
            all_genes = sorted(set.union(*gene_sets))
            expr_list_aligned = [df.reindex(all_genes) for df in expr_list]

        # Concatenar
        expr_mega = pd.concat(expr_list_aligned, axis=1)

        # Metadata
        meta_mega = pd.concat(meta_list, axis=0, ignore_index=True)
        meta_mega = meta_mega.drop_duplicates(subset=["sample"], keep="last")
        cols_present = set(expr_mega.columns)
        meta_mega = meta_mega[meta_mega["sample"].isin(cols_present)].copy()

        front = [c for c in ["sample", "batch", "label"] if c in meta_mega.columns]
        rest  = [c for c in meta_mega.columns if c not in front]
        meta_mega = meta_mega[front + rest]

        print(f" ✓ Grupo '{group_name}': Genes={expr_mega.shape[0]}, Muestras={expr_mega.shape[1]}, Batches={meta_mega['batch'].nunique()}")
        return expr_mega, meta_mega

    expr_counts_mega, meta_counts_mega = _align_concat(expr_counts_list, meta_counts_list, "counts")
    expr_norm_mega,   meta_norm_mega   = _align_concat(expr_norm_list,   meta_norm_list,   "normalized")

    return expr_counts_mega, meta_counts_mega, expr_norm_mega, meta_norm_mega, scale_report


In [15]:
counts_symbol_rnaseq, rnaseq_meta, counts_symbol_rnaseq_norm, rnaseq_meta_norm, scale_report = \
    build_split_mega_tables_from_globals(
        globals(),
        expr_prefix="counts_symbol_",
        prefer_intersection=True,
        min_common_genes_warn=100,
        unknown_to="normalized"   # <- manda los 'unknown' a COUNTS unknown_to="normalized"
    )

Detectados estudios: GSE96058, TCGA
 · GSE96058: normalized (hay valores negativos (típico VST/rlog))
 · TCGA: normalized (rango pequeño y decimales (log/transformado))
 · No hay estudios en grupo 'counts'.
 ✓ Grupo 'normalized': Genes=18603, Muestras=4313, Batches=2


In [16]:
scale_report

,study_id,varname,meta_used,meta_overlap,group,reason,nonneg,frac_integers,sum_cv,q95,max,zero_frac
0,GSE96058,counts_symbol_GSE96058,GSE96058_meta,1.0,normalized,hay valores negativos (típico VST/rlog),False,0.000001,-5.560650,5.579263,25.000093,4.752003e-08
1,TCGA,counts_symbol_TCGA,TCGA_meta,1.0,normalized,rango pequeño y decimales (log/transformado),True,0.142798,0.022674,11.764700,20.978400,1.427093e-01


In [114]:
#del counts_symbol_rnaseq, rnaseq_meta, counts_symbol_rnaseq_norm, rnaseq_meta_norm, scale_report

In [31]:
counts_symbol_GSE96058

F1         F2         F3  \
gene_symbol                                                                 
5S_rRNA                                    4.911099  -3.321928  -3.321928   
5_8S_rRNA                                 -3.321928  -3.321928  -3.321928   
6M1-18                                    -3.321928  -3.321928  -3.321928   
7M1-2                                     -3.321928  -3.321928  -3.321928   
7SK                                       -0.539253  -0.576620  -1.651323   
...                                             ...        ...        ...   
septin 9/TNRC6C fusion                    12.397228  -3.321928  -3.321928   
svRNAb                                    -3.321928  -3.321928  -3.321928   
tRNA(Ile)                                 12.437523  12.726540  13.095962   
unknown                                   -1.167750   2.579381   1.254933   
vitamin D 1-alpha-hydroxylase splice ...  -3.321928  -3.321928  -3.321928   

                                                 F4         F5         F6  \
gene_symbol                                                                 
5S_rRNA                                    3.656393   4.190104   2.556304   
5_8S_rRNA                                 -3.321928  -3.321928  -3.321928   
6M1-18                                    -3.321928  -3.321928  -3.321928   
7M1-2                                     -3.321928  -3.321928  -3.321928   
7SK                                        0.126633   0.783715  -1.759556   
...                                             ...        ...        ...   
septin 9/TNRC6C fusion                    -3.321928  -3.321928  -3.321928   
svRNAb                                    -3.321928  -3.321928  -3.321928   
tRNA(Ile)                                 12.423323  12.198522  12.574657   
unknown                                    2.128970   3.414582   2.233232   
vitamin D 1-alpha-hydroxylase splice ...  -3.321928  -0.311308  -3.321928   

                                                 F7         F8         F9  \
gene_symbol                                                                 
5S_rRNA                                    2.590351   5.691788  -3.321928   
5_8S_rRNA                                 -3.321928  -3.321928  -3.321928   
6M1-18                                    -3.321928  -3.321928  -3.321928   
7M1-2                                     -3.321928  -3.321928  -3.321928   
7SK                                       -1.033968  -0.129513   0.494308   
...                                             ...        ...        ...   
septin 9/TNRC6C fusion                     9.792650  10.515522  -3.321928   
svRNAb                                    -3.321928  -3.321928  -3.321928   
tRNA(Ile)                                 13.320519  12.836467  12.308040   
unknown                                    2.567297   2.646835   2.767061   
vitamin D 1-alpha-hydroxylase splice ...  -3.321928  -3.321928  -2.385246   

                                                F10        F11        F12  \
gene_symbol                                                                 
5S_rRNA                                    3.223401  -3.321928   5.265988   
5_8S_rRNA                                 -3.321928  -3.321928  -3.321928   
6M1-18                                    -3.321928  -3.321928  -3.321928   
7M1-2                                     -3.321928  -3.321928  -3.321928   
7SK                                        0.749841   1.541830   0.049312   
...                                             ...        ...        ...   
septin 9/TNRC6C fusion                    -3.321928  -3.321928  -3.321928   
svRNAb                                    -3.321928  -3.321928  -3.321928   
tRNA(Ile)                                 13.807979  13.143538  13.909762   
unknown                                    2.292104   3.097019   1.948298   
vitamin D 1-alpha-hydroxylase splice ...  -1.940623  -0.251896  -3.319201   

                                         

0.445114992494126

In [18]:
# TOTAL de NaN en todo el DataFrame
total_nans = counts_symbol_rnaseq_norm.isna().sum().sum()

# NaN POR COLUMNA (ordenado de mayor a menor)
nans_por_col = counts_symbol_rnaseq_norm.isna().sum().sort_values(ascending=False)

# Porcentaje de NaN POR COLUMNA
pct_nans_por_col = (counts_symbol_rnaseq_norm.isna().mean() * 100).sort_values(ascending=False)

# Número de columnas que tienen al menos un NaN
num_cols_con_nan = counts_symbol_rnaseq_norm.isna().any().sum()

# Columnas que tienen NaN
cols_con_nan = counts_symbol_rnaseq_norm.columns[counts_symbol_rnaseq_norm.isna().any()].tolist()

# NaN POR FILA
nans_por_fila = counts_symbol_rnaseq_norm.isna().sum(axis=1)

print("Total NaN:", total_nans)
print("Columnas con NaN:", num_cols_con_nan)
print("Top columnas con NaN:\n", nans_por_col.head())

Total NaN: 0
Columnas con NaN: 0
Top columnas con NaN:
 TCGA:TCGA-BH-A1EV-11    0
GSE96058:F1             0
GSE96058:F2             0
GSE96058:F3             0
GSE96058:F4             0
dtype: int64


In [21]:
#counts_symbol_rnaseq.to_csv("../Data/counts_symbol_rnaseq.csv")
counts_symbol_rnaseq_norm.to_csv("./Data/counts_symbol_rnaseq_norm.csv")
#rnaseq_meta.to_csv("../Data/rnaseq_meta.csv", index=False)
rnaseq_meta_norm.to_csv("./Data/rnaseq_meta_norm.csv", index=False)

In [25]:
counts_symbol_rnaseq_norm.sample(n=300, random_state=42).to_csv("./Data/counts_symbol_rnaseq_norm_muestra.csv")
rnaseq_meta_norm.sample(n=300, random_state=42).to_csv("./Data/rnaseq_meta_norm_muestra.csv")

In [124]:
GSE96058_meta

,label,batch,sample
GSM,,,
GSM2528079,TNBC,GSE96058,GSM2528079
GSM2528080,LumA,GSE96058,GSM2528080
GSM2528081,LumB,GSE96058,GSM2528081
GSM2528082,LumA,GSE96058,GSM2528082
GSM2528083,Normal,GSE96058,GSM2528083
...,...,...,...
GSM2531483,LumB,GSE96058,GSM2531483
GSM2531484,HER2,GSE96058,GSM2531484
GSM2531485,TNBC,GSE96058,GSM2531485


In [24]:
counts_symbol_rnaseq_norm.min()

GSE96058:F1            -3.321928
GSE96058:F2            -3.321928
GSE96058:F3            -3.321928
GSE96058:F4            -3.321928
GSE96058:F5            -3.321928
                          ...   
TCGA:TCGA-E9-A5FL-01    0.000000
TCGA:TCGA-AC-A2FB-11    0.000000
TCGA:TCGA-A2-A3XT-01    0.000000
TCGA:TCGA-B6-A0X7-01    0.000000
TCGA:TCGA-BH-A1EV-11    0.000000
Length: 4313, dtype: float64